# 01 — Powerful Spark 4.x Features

This notebook demonstrates:

1. SparkSession + Adaptive Query Execution (AQE) + Dynamic Partition Pruning (DPP)
2. Pandas API on Spark
3. Delta Lake — Time Travel + Optimize/Z-Order (if libraries are available)
4. Structured Streaming — `foreachBatch` + `Trigger.AvailableNow`
5. Apache Iceberg — schema evolution + hidden partitioning (if libraries are available)
6. RAPIDS Accelerator — GPU check/demo (if plugin is available)
7. Spark Connect — remote DataFrame API (if server is available)
8. MLlib Pipeline + optional ONNX export

Optional sections are wrapped in `try/except`, so the notebook continues even without Delta/Iceberg/RAPIDS/Spark Connect/ONNX dependencies.

## Cell 1 — AQE + DPP sanity demo

Simulates a partitioned fact table joined with a small dimension to trigger runtime optimizations.

The broadcast join enables Dynamic Partition Pruning, limiting scanned partitions to only relevant keys, while AQE adjusts execution based on actual data flow (e.g. join strategy, shuffle behavior).

The explain plan is used to verify that these optimizations are applied.

In [1]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
DATA_DIR = Path(os.environ.get("DATA_DIR", "/workspace/data"))
WAREHOUSE_DIR = os.environ.get("SPARK_WAREHOUSE_DIR", "/workspace/spark-warehouse")
GLUTEN_ENABLED = os.environ.get("GLUTEN_ENABLED", "false").lower() == "true"

DATA_DIR.mkdir(parents=True, exist_ok=True)

builder = (
    SparkSession.builder
    .appName("powerful-spark-4x-features")
    .master(MASTER)
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.cores", "2")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.sql.dynamicPartitionPruning.enabled", "true")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.execution.datasources.v2.V2SessionCatalog")
    .config("spark.sql.catalog.local_iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local_iceberg.type", "hadoop")
    .config("spark.sql.catalog.local_iceberg.warehouse", f"{WAREHOUSE_DIR}/iceberg")
)

if GLUTEN_ENABLED:
    builder = (
        builder
        .config("spark.plugins", "org.apache.gluten.GlutenPlugin")
        .config("spark.memory.offHeap.enabled", "true")
        .config("spark.memory.offHeap.size", "2g")
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("DPP enabled:", spark.conf.get("spark.sql.dynamicPartitionPruning.enabled"))

Spark version: 4.0.2
Spark master: spark://spark-master:7077
AQE enabled: true
DPP enabled: true


In [2]:
# Small AQE + DPP sanity demo
base_path = DATA_DIR / "aqe_dpp_demo"
shutil.rmtree(base_path, ignore_errors=True)

fact = (
    spark.range(0, 1_000_000)
    .withColumn("day", (F.col("id") % 30).cast("int"))
    .withColumn("customer_id", (F.col("id") % 10_000).cast("int"))
    .withColumn("amount", (F.rand(seed=42) * 100).cast("double"))
)

fact.write.mode("overwrite").partitionBy("day").parquet(str(base_path / "fact_sales"))

dim_days = spark.createDataFrame([(1, "promo"), (7, "promo"), (14, "promo")], ["day", "campaign"])
fact_sales = spark.read.parquet(str(base_path / "fact_sales"))

filtered = (
    fact_sales
    .join(F.broadcast(dim_days), "day")
    .groupBy("campaign")
    .agg(F.count("*").alias("rows"), F.round(F.sum("amount"), 2).alias("total_amount"))
)

filtered.show()
filtered.explain("formatted")

[Stage 3:>                                                          (0 + 2) / 2]

+--------+------+------------+
|campaign|  rows|total_amount|
+--------+------+------------+
|   promo|100001|  4993405.67|
+--------+------+------------+

== Physical Plan ==
AdaptiveSparkPlan (10)
+- HashAggregate (9)
   +- Exchange (8)
      +- HashAggregate (7)
         +- Project (6)
            +- BroadcastHashJoin Inner BuildRight (5)
               :- Scan parquet  (1)
               +- BroadcastExchange (4)
                  +- Filter (3)
                     +- Scan ExistingRDD (2)


(1) Scan parquet 
Output [2]: [amount#8, day#9]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/aqe_dpp_demo/fact_sales]
PartitionFilters: [isnotnull(day#9)]
ReadSchema: struct<amount:double>

(2) Scan ExistingRDD
Output [2]: [day#4L, campaign#5]
Arguments: [day#4L, campaign#5], MapPartitionsRDD[8] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(3) Filter
Input [2]: [day#4L, campaign#5]
Condition : isnotnull(day#4L)

(4) Broadcas

## Cell 2 — Pandas API on Spark

Demonstrates pandas-style data processing executed on Spark without rewriting logic into native Spark APIs.

Operations like groupby and aggregation are translated into distributed Spark execution, allowing familiar pandas workflows to scale transparently.

In [3]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

try:
    import numpy as np
    import pyspark.pandas as ps

    ps.set_option("compute.default_index_type", "distributed")

    pdf = ps.DataFrame({
        "id": range(1, 100_001),
        "value": np.random.default_rng(42).normal(size=100_000),
        "category": np.random.default_rng(7).choice(["A", "B", "C", "D"], size=100_000),
    })

    result = (
        pdf.groupby("category")
        .agg(mean_val=("value", "mean"), cnt=("value", "count"))
        .sort_values("mean_val", ascending=False)
    )

    print(result.head(10))
except Exception as e:
    print("Pandas API on Spark is not available in this environment:", e)

26/04/26 06:47:15 WARN TaskSetManager: Stage 6 contains a task of very large size (2119 KiB). The maximum recommended task size is 1000 KiB.
          mean_val    cnt
category                 
C         0.001508  25034
D         0.000822  25087
B        -0.005220  24981
A        -0.014108  24898


## Cell 3 — Delta Lake: Time Travel & Optimize/Z-Order

Shows how Delta maintains versioned table state, enabling consistent reads of past data snapshots alongside current data.

The focus is on separating logical table history from physical data changes, allowing reproducibility and safe incremental updates, while optional optimizations (like Z-Order) depend on runtime support.

In [4]:
def has_data_source(fmt: str) -> bool:
    probe_path = DATA_DIR / f"_probe_{fmt}"

    try:
        shutil.rmtree(probe_path, ignore_errors=True)
        spark.range(1).write.format(fmt).mode("overwrite").save(str(probe_path))
        return True
    except Exception:
        return False
    finally:
        shutil.rmtree(probe_path, ignore_errors=True)


try:
    delta_path = DATA_DIR / "delta" / "events"
    shutil.rmtree(delta_path, ignore_errors=True)

    if not has_data_source("delta"):
        raise RuntimeError("Delta Lake source is not on the Spark classpath.")

    df_delta = (
        spark.range(0, 1_000_000)
        .withColumn("value", F.rand(seed=42) * 100)
        .withColumn("day", (F.rand(seed=7) * 30).cast("int"))
        .withColumn(
            "category",
            F.when(F.rand(seed=11) < 0.33, F.lit("X"))
             .when(F.rand(seed=13) < 0.66, F.lit("Y"))
             .otherwise(F.lit("Z"))
        )
    )

    (
        df_delta.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("day")
        .save(str(delta_path))
    )

    version_0_count = (
        spark.read
        .format("delta")
        .option("versionAsOf", 0)
        .load(str(delta_path))
        .count()
    )

    (
        spark.range(1_000_000, 1_010_000)
        .withColumn("value", F.rand(seed=99) * 100)
        .withColumn("day", F.lit(1))
        .withColumn("category", F.lit("X"))
        .write
        .format("delta")
        .mode("append")
        .save(str(delta_path))
    )

    current_count = spark.read.format("delta").load(str(delta_path)).count()

    print("Delta version 0 rows:", version_0_count)
    print("Delta current rows:", current_count)

    try:
        spark.sql(f"OPTIMIZE delta.`{str(delta_path)}` ZORDER BY (value)").show(truncate=False)
    except Exception as optimize_error:
        print("OPTIMIZE/ZORDER skipped: not supported by this local Delta runtime.")

except Exception as e:
    print("Delta Lake demo skipped:", e)

26/04/26 06:47:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta version 0 rows: 1000000
Delta current rows: 1010000
OPTIMIZE/ZORDER skipped: not supported by this local Delta runtime.


## Cell 4 — Structured Streaming: `foreachBatch` + `Trigger.AvailableNow`

Runs a finite Structured Streaming job using the `rate` source and a Parquet sink, so it works even without Delta Lake.

The important part is the `foreachBatch` pattern: each micro-batch is handled as a normal DataFrame, which is the usual bridge between streaming ingestion and batch-style ETL writes.

In this local demo, Spark may warn that AQE is disabled for streaming and that the `rate` source falls back from `AvailableNow` to a single-batch execution. That is expected for this source and does not break the example.

In [5]:
stream_path = DATA_DIR / "stream_action_counts"
checkpoint_path = DATA_DIR / "checkpoints" / "stream_action_counts"
shutil.rmtree(stream_path, ignore_errors=True)
shutil.rmtree(checkpoint_path, ignore_errors=True)

raw = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 1000)
    .option("numPartitions", 2)
    .load()
)

events = raw.select(
    F.concat(F.lit("user_"), (F.col("value") % 100).cast("string")).alias("user_id"),
    F.when((F.col("value") % 3) == 0, F.lit("click"))
     .when((F.col("value") % 3) == 1, F.lit("view"))
     .otherwise(F.lit("purchase")).alias("action"),
    F.col("timestamp").alias("ts")
)

windowed = (
    events
    .withWatermark("ts", "10 minutes")
    .groupBy(F.window("ts", "1 minute"), "action")
    .agg(F.count("*").alias("cnt"))
)

def foreach_batch_func(batch_df, batch_id):
    (
        batch_df
        .withColumn("batch_id", F.lit(batch_id))
        .write
        .mode("append")
        .parquet(str(stream_path))
    )

query = (
    windowed.writeStream
    .foreachBatch(foreach_batch_func)
    .outputMode("update")
    .trigger(availableNow=True)
    .option("checkpointLocation", str(checkpoint_path))
    .start()
)

query.awaitTermination()
print("Streaming query completed.")

if stream_path.exists():
    spark.read.parquet(str(stream_path)).orderBy("batch_id", "action").show(20, truncate=False)
else:
    print("No streaming output was produced. Re-run the cell after a short delay if needed.")

26/04/26 06:47:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/26 06:47:34 WARN MicroBatchExecution: source [RateStreamV2[rowsPerSecond=1000, rampUpTimeSeconds=0, numPartitions=2] does not support Trigger.AvailableNow. Falling back to single batch execution. Note that this may not guarantee processing new data if there is an uncommitted batch. Please consult with data source developer to support Trigger.AvailableNow.


Streaming query completed.
+------+------+---+--------+
|window|action|cnt|batch_id|
+------+------+---+--------+
+------+------+---+--------+



## Cell 5 — Apache Iceberg: Schema Evolution & Hidden Partitioning

Demonstrates how Iceberg separates table metadata from physical data layout, enabling schema changes without rewriting existing data.

Hidden partitioning allows efficient partition pruning without exposing partition columns to the user, simplifying queries while retaining performance benefits.

Runs only if the Iceberg runtime is available on the Spark classpath.

In [ ]:
try:
    spark.sql("CREATE DATABASE IF NOT EXISTS local_iceberg.demo")
    spark.sql("DROP TABLE IF EXISTS local_iceberg.demo.events_iceberg")

    spark.sql("""
    CREATE TABLE local_iceberg.demo.events_iceberg (
        event_id STRING,
        user_id STRING,
        event_ts TIMESTAMP,
        action STRING,
        dt DATE
    )
    USING iceberg
    PARTITIONED BY (days(dt))
    """)

    (
        spark.range(1, 100_001)
        .selectExpr(
            "cast(id as string) as event_id",
            "concat('user_', cast((id % 1000) as string)) as user_id",
            "timestampadd(SECOND, cast(id as int), current_timestamp()) as event_ts",
            "case when (id % 3) = 0 then 'click' when (id % 3) = 1 then 'view' else 'purchase' end as action",
            "date(current_timestamp()) as dt"
        )
        .writeTo("local_iceberg.demo.events_iceberg")
        .append()
    )

    spark.sql("ALTER TABLE local_iceberg.demo.events_iceberg ADD COLUMN is_fraud BOOLEAN")

    spark.sql("""
    SELECT dt, action, COUNT(*) AS rows
    FROM local_iceberg.demo.events_iceberg
    WHERE action = 'purchase'
    GROUP BY dt, action
    ORDER BY dt, action
    """).show(truncate=False)

    spark.sql("DESCRIBE TABLE local_iceberg.demo.events_iceberg").show(truncate=False)
except Exception as e:
    print("Iceberg demo skipped:", e)

## Cell 6 — RAPIDS GPU Accelerator demo

Checks whether the RAPIDS plugin is active and runs a simple aggregation to illustrate GPU offloading.

If the plugin is not available, the same logic runs on CPU, highlighting that GPU acceleration is transparent and does not require changes to the code.

In [ ]:
rapids_plugin = spark.conf.get("spark.plugins", "")
rapids_enabled = "RapidsPlugin" in rapids_plugin or "com.nvidia.spark.SQLPlugin" in rapids_plugin

print("spark.plugins:", rapids_plugin or "<not set>")
print("RAPIDS detected:", rapids_enabled)

if rapids_enabled:
    print("RAPIDS accelerator detected — check Spark UI SQL tab for GPU operators/metrics.")
else:
    print("RAPIDS is not enabled in this session — running CPU-safe version.")

df_gpu_demo = (
    spark.range(0, 5_000_000)
    .withColumn("val", (F.col("id") * F.lit(123456789)) % F.lit(1000))
    .withColumn("grp", F.col("id") % F.lit(100))
)

result_gpu_demo = df_gpu_demo.groupBy("grp").agg(
    F.sum("val").alias("sum_val"),
    F.avg("id").alias("avg_id")
).orderBy("grp")

result_gpu_demo.show(10)

## Cell 7 — Spark Connect: Remote DataFrame API

Demonstrates how a client can execute DataFrame operations against a remote Spark cluster via Spark Connect.

The focus is on decoupling the client environment from the execution environment — the same code runs locally while computation happens remotely.

If the Spark Connect server is not available, the section is skipped.

In [3]:
import socket

try:
    from pyspark.sql.connect.session import SparkSession as ConnectSession

    remote_url = "sc://spark-connect:15002"

    host = "spark-connect"
    port = 15002

    with socket.create_connection((host, port), timeout=5):
        print(f"Spark Connect port is reachable: {remote_url}")

    cs = (
        ConnectSession.builder
        .remote(remote_url)
        .appName("spark-connect-demo")
        .getOrCreate()
    )

    remote_count = cs.range(0, 1000).where("id % 2 = 0").count()

    print("Spark Connect remote:", remote_url)
    print("Remote even rows:", remote_count)

    cs.stop()

except Exception as e:
    print("Spark Connect demo skipped:", e)

Spark Connect port is reachable: sc://spark-connect:15002
Spark Connect remote: sc://spark-connect:15002
Remote even rows: 500


In [4]:
import socket
import time

try:
    from pyspark.sql.connect.session import SparkSession as ConnectSession

    remote_url = "sc://spark-connect:15002"

    with socket.create_connection(("spark-connect", 15002), timeout=5):
        print(f"Spark Connect port is reachable: {remote_url}")

    cs = (
        ConnectSession.builder
        .remote(remote_url)
        .appName("spark-connect-demo")
        .config("spark.sql.shuffle.partitions", "1")
        .getOrCreate()
    )

    print("Spark Connect session created")

    df = cs.sql("SELECT 1 AS ok")
    df.show()

    cs.stop()
    print("Spark Connect demo completed")

except Exception as e:
    print("Spark Connect demo skipped:", e)

Spark Connect port is reachable: sc://spark-connect:15002
Spark Connect session created
+---+
| ok|
+---+
|  1|
+---+

Spark Connect demo completed


In [2]:
from pyspark.sql.connect.session import SparkSession as ConnectSession

cs = (
    ConnectSession.builder
    .remote("sc://spark-connect:15002")
    .appName("spark-connect-smoke-test")
    .getOrCreate()
)

cs.sql("SELECT 1 AS ok").show()

cs.stop()

+---+
| ok|
+---+
|  1|
+---+



## Cell 8 — MLlib Pipeline + optional ONNX Export

End-to-end pipeline: categorical indexing, one-hot encoding, vector assembly, scaling, classifier, evaluator. ONNX export je voliteľný a preskočí sa, ak chýba balík.

In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

ml_df = (
    spark.range(0, 50_000)
    .withColumn("user_id", F.concat(F.lit("user_"), (F.col("id") % 1000).cast("string")))
    .withColumn("action", F.when((F.col("id") % 3) == 0, "click").when((F.col("id") % 3) == 1, "view").otherwise("purchase"))
    .withColumn("feature_val", (F.rand(seed=42) * 100).cast("double"))
    .withColumn("label", ((F.col("id") % 17 == 0) | ((F.col("action") == "purchase") & (F.col("feature_val") > 80))).cast("double"))
    .select("user_id", "action", "feature_val", "label")
)

user_idx = StringIndexer(inputCol="user_id", outputCol="user_idx", handleInvalid="keep")
action_idx = StringIndexer(inputCol="action", outputCol="action_idx", handleInvalid="keep")
user_enc = OneHotEncoder(inputCol="user_idx", outputCol="user_vec")
action_enc = OneHotEncoder(inputCol="action_idx", outputCol="action_vec")
assembler = VectorAssembler(inputCols=["user_vec", "action_vec", "feature_val"], outputCol="rawFeatures")
scaler = StandardScaler(inputCol="rawFeatures", outputCol="features", withStd=True, withMean=False)
gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=10, seed=42)

pipeline = Pipeline(stages=[user_idx, action_idx, user_enc, action_enc, assembler, scaler, gbt])

param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.maxIter, [5, 10])
    .build()
)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction")
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42,
)

train, test = ml_df.randomSplit([0.8, 0.2], seed=42)
cv_model = cv.fit(train)
pred = cv_model.transform(test)
auc = evaluator.evaluate(pred)

print(f"Test AUC: {auc:.4f}")
pred.select("user_id", "action", "feature_val", "label", "prediction", "probability").show(10, truncate=False)

try:
    from sparkonnx import convert_model

    onnx_bytes = convert_model(cv_model.bestModel, spark)
    onnx_path = DATA_DIR / "fraud_gbt_model.onnx"
    with open(onnx_path, "wb") as f:
        f.write(onnx_bytes)
    print("Model exported to:", onnx_path)
except Exception as e:
    print("ONNX export skipped. Install/configure sparkonnx if needed:", e)

Test AUC: 0.7687
+-------+--------+------------------+-----+----------+----------------------------------------+
|user_id|action  |feature_val       |label|prediction|probability                             |
+-------+--------+------------------+-----+----------+----------------------------------------+
|user_0 |click   |16.372590299182843|0.0  |0.0       |[0.8816930045639652,0.11830699543603485]|
|user_0 |click   |61.918937022530095|1.0  |0.0       |[0.8796098638720782,0.12039013612792182]|
|user_0 |click   |81.95576318675197 |0.0  |0.0       |[0.8849217505590418,0.11507824944095824]|
|user_0 |purchase|49.706647232307034|0.0  |0.0       |[0.8816930045639652,0.11830699543603485]|
|user_0 |view    |23.46812187986014 |0.0  |0.0       |[0.8816930045639652,0.11830699543603485]|
|user_0 |view    |86.91244995872881 |0.0  |0.0       |[0.8849217505590418,0.11507824944095824]|
|user_1 |click   |75.6754367657105  |0.0  |0.0       |[0.8796098638720782,0.12039013612792182]|
|user_1 |purchase|76.50

## Final sanity check

In [6]:
print("Spark app:", spark.sparkContext.appName)
print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
spark.range(5).show()

Spark app: powerful-spark-4x-features
Spark version: 4.0.2
Default parallelism: 2
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+

